In [0]:
spark

In [0]:
from pyspark.sql import SparkSession
spark1 = SparkSession.builder.appName('Batch28').getOrCreate()
spark1

In [0]:
%sql
show catalogs

In [0]:
%sql
use catalog batch28_catalog

In [0]:
%sql


In [0]:
%sql
CREATE TABLE IF NOT EXISTS batch28_catalog.default.employee (
    emp_id        INT          NOT NULL,
    first_name    STRING,
    last_name     STRING,
    email         STRING,
    hire_date     DATE,
    salary        DECIMAL(10,2),
    department_id INT
);

In [0]:
%sql
show schemas

In [0]:
%sql
use schema test_batch28_schema

In [0]:
%sql
CREATE TABLE IF NOT EXISTS test_batch28_schema.employee (
    emp_id        INT          NOT NULL,
    first_name    STRING,
    last_name     STRING,
    email         STRING,
    hire_date     DATE,
    salary        DECIMAL(10,2),
    department_id INT
);

In [0]:
%sql
show volumes

In [0]:
dbutils.fs.ls('/')

In [0]:
#####Use in azure check path
dbutils.fs.ls('batch28_catalog.test_batch28_schema.bronze')

In [0]:
#######Read file  copy path in volummes#######
path = r"/Volumes/batch28_catalog/test_batch28_schema/bronze/customers-2000000.csv"
path
df = spark.read.csv(path,header = True,inferSchema = True)
#df = spark.read.csv(path,header = True,enforceSchema= True)
df.show(truncate=False)
df.display()
df.printSchema()

In [0]:
#####columns names
df.columns

In [0]:
#count rows count
df.count()

In [0]:
#columns name count find
print(len(df.columns))

In [0]:
df.dtypes

In [0]:
#find intger and string columns
str_col = []
int_col = []
for col_name in df.dtypes:
    if col_name[1]=='int':
        int_col.append(col_name[0])
    elif col_name[1]=='string':
        str_col.append(col_name[0])
    else:
        print(f'columns name:{col_name[0]} Not str and int')
print(f"string columns{str_col}")
print(f"interger columns{int_col}")

In [0]:
df1 = df.select('Customer Id', 'First Name', 'Last Name')
df1.display()

In [0]:
df1 = df[['Customer Id', 'First Name', 'Last Name']]
df1.display()

In [0]:
df.display()

In [0]:
#columns rename and lower
df2 = df
for col_name in df.columns:
    df2 = df2.withColumnRenamed(col_name, col_name.replace(' ','_').lower())

df2.display()

In [0]:
#columns name rename
from pyspark.sql.functions import *
df3 = df2.withColumnRenamed('index','in')
df3.display()

df3 = df3.drop('in')
df3.display()

In [0]:
df3 = df3.withColumn("Owner",lit('Batch28'))
df3.display()

In [0]:
df3 = df3.withColumn('Current_Date',current_date())\
    .withColumn('Time',current_timestamp())
df3.display()

In [0]:
#subscription_date ko year date and day column m splot krna hai

df3 = df3.withColumn("year", year(df3.subscription_date))\
        .withColumn("day", dayofmonth(df3.subscription_date))\
        .withColumn("Month",month(df3.subscription_date))


df3.display()

In [0]:
path = '/Volumes/batch28_catalog/test_batch28_schema/bronze/weather.parquet'
df = spark.read.parquet(path)
df.display()

In [0]:
########herder add in files
from pyspark.sql.types import *
path = '/Volumes/batch28_catalog/test_batch28_schema/bronze/batch18_student_without_header.csv'

# schema = StructType([
#     StructField("Customer Id", StringType(), True),
#     StructField("First Name", StringType(), True),
#     StructField("Last Name", StringType(), True),
#     StructField("Company", StringType(), True),
#     Struct

schema = StructType([
    StructField('ID',StringType(),True),
    StructField('Name',StringType(),True),
    StructField('Age',IntegerType(),True),
    StructField('Salary',DoubleType(),True),
])

df = spark.read.csv(path,schema = schema)
df.display()







In [0]:
from pyspark.sql.functions import *

df1 = df.withColumn('updated_age',col('Age')+10)\
         .drop('Age')\
        .withColumnRenamed('updated_age','Age')
df1.display()

In [0]:
#write the save the file
path = '/Volumes/batch28_catalog/test_batch28_schema/silver/Gaurav'
df1.write.mode('overwrite').csv(path)

In [0]:
#only for 1 partition
path = '/Volumes/batch28_catalog/test_batch28_schema/silver/Gaurav'
df1.coalesce(1).write.mode('append').option('herder','true').csv(path)


In [0]:
############Window function#############

from pyspark.sql.window import *
from pyspark.sql.functions import *
path = '/Volumes/batch28_catalog/test_batch28_schema/bronze/batch14_window.csv'
df1 = spark.read.csv(path,header = True,inferSchema = True)
df1.display()

In [0]:
from pyspark.sql.window import *
emp_window = Window.partitionBy('depart').orderBy('salary')

df2 = df1.withColumn('row_num',row_number().over(emp_window))\
    .withColumn('ran',rank().over(emp_window))\
    .withColumn('dense',dense_rank().over(emp_window))
df2.display()

In [0]:
df3 = df1.withColumn('lead',lead('id').over(emp_window))\
    .withColumn('lg',lag('id').over(emp_window))
df3.display()
# df4 = df1.withColumn('sum',sum('id').over(emp_window))\
#     .withColumn('avg',avg('id').over(emp_window))\
#     .withColumn('min',min('id').over(emp_window))\
#     .withColumn('max',max('id').over(emp_window))
# df4.display()